In [12]:
!pip install skyfield



In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from skyfield.api import load
from tqdm import tqdm
import itertools


In [14]:
FEATURES_TO_USE = [
    'mean_motion_rev_day',
    'eccentricity',
    'bstar_drag',
    'inclination_deg',
    'raan_deg',
    'arg_perigee_deg',
    'mean_anomaly_deg'
]

In [15]:

class HybridDatasetGenerator:

    def __init__(self, satellites, df_parsed):
        self.satellites = satellites
        self.df_parsed = df_parsed

    def generate_single(self, feature_group):
        """
        Create a hybrid dataset for ANY number of features.
        feature_group: tuple or list of columns
        """

        hybrid_records = []

        for i in range(1, len(self.satellites)):

            sat_t0 = self.satellites[i-1]
            sat_t1 = self.satellites[i]

            predicted = sat_t0.at(sat_t1.epoch).position.km
            actual    = sat_t1.at(sat_t1.epoch).position.km
            error_vec = actual - predicted

            row = self.df_parsed.loc[i-1]

            record = {}

            for feat in feature_group:
                record[feat] = row[feat]

            record['delta_t_days'] = sat_t1.epoch.tt - sat_t0.epoch.tt

            record['error_x_km'] = error_vec[0]
            record['error_y_km'] = error_vec[1]
            record['error_z_km'] = error_vec[2]

            hybrid_records.append(record)

        df_hybrid = pd.DataFrame(hybrid_records)
        return df_hybrid.dropna()

    def generate_combinations(self, feature_list, k):
        """
        Automatically create ALL datasets for combinations of size k.
        Returns a dict: {feature_tuple: dataframe}
        """

        combos = list(itertools.combinations(feature_list, k))
        results = {}

        print(f"\nGenerating datasets for {len(combos)} combinations of size {k}")

        for comb in combos:
            print(f" -> {comb}")
            df_h = self.generate_single(comb)
            results[comb] = df_h

        return results

    def save_all(self, results_dict, output_folder="/content/"):
        """
        Save all generated datasets to CSV.
        """

        for feats, df in results_dict.items():
            name = "_".join(feats)
            path = f"{output_folder}/hybrid_{name}.csv"
            df.to_csv(path, index=False)
            print(f"Saved: {path} ({len(df)} rows)")


In [16]:


TLE_FILENAME = "tle_history_38054.txt"
PARSED_DATA_CSV = "parsed_debris_data.csv"
OUTPUT_HYBRID_CSV = "hybrid_training_data.csv"


df_parsed = pd.read_csv(PARSED_DATA_CSV)



In [17]:
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from skyfield.api import load
from tqdm import tqdm
import itertools
from tensorflow.keras.losses import MeanSquaredError

def train_hybrid_lstm(
        csv_path,
        feature_list,
        target_list=['error_x_km','error_y_km','error_z_km'],
        lookback=30,
        train_split=0.8,
        model_name_prefix="hybrid_lstm"
    ):

    print(f"\n=== Training model for features: {feature_list} ===")
    print(f"Loading dataset: {csv_path}")

    df = pd.read_csv(csv_path).dropna()

    split_idx = int(len(df) * train_split)
    df_train = df.iloc[:split_idx]
    df_test  = df.iloc[split_idx:]

    feat_scaler = MinMaxScaler()
    tgt_scaler = MinMaxScaler()

    feat_scaler.fit(df_train[feature_list])
    tgt_scaler.fit(df_train[target_list])

    X_train_scaled = feat_scaler.transform(df_train[feature_list])
    y_train_scaled = tgt_scaler.transform(df_train[target_list])

    X_test_scaled = feat_scaler.transform(df_test[feature_list])
    y_test_scaled = tgt_scaler.transform(df_test[target_list])

    def create_sequences(features, targets, lookback):
        X, y = [], []
        for i in range(len(features) - lookback):
            X.append(features[i:(i+lookback)])
            y.append(targets[i+lookback])
        return np.array(X), np.array(y)

    X_train, y_train = create_sequences(X_train_scaled, y_train_scaled, lookback)
    X_test, y_test   = create_sequences(X_test_scaled, y_test_scaled, lookback)

    model = Sequential()
    model.add(Bidirectional(LSTM(75, input_shape=(lookback, len(feature_list)))))
    model.add(Dropout(0.3))
    model.add(Dense(25))
    model.add(Dense(len(target_list)))
    model.compile(optimizer="adam", loss=MeanSquaredError())

    early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_test, y_test),
        shuffle=False,
        callbacks=[early_stop],
        verbose=1
    )

    fname = f"{model_name_prefix}_{'_'.join(feature_list)}.h5"
    model.save(fname)
    print(f"Saved model as {fname}")

    final_loss = history.history["val_loss"][-1]
    print(f"Final val loss = {final_loss}")

    return final_loss, fname

In [18]:
feature_combinations = list(itertools.combinations(FEATURES_TO_USE,7))

In [19]:
def evaluate_model(feature_list,
                   hybrid_csv_path,
                   model_path,
                   output_csv_path,
                   lookback=30,
                   train_split_ratio=0.8):
    """
    Evaluates a trained LSTM residual model for ANY given feature subset.
    """

    print("\n" + "="*80)
    print(f" Evaluating Model Using Features: {feature_list}")
    print("="*80)

    print(f"Loading hybrid dataset: {hybrid_csv_path}")
    df = pd.read_csv(hybrid_csv_path)

    split_idx = int(len(df) * train_split_ratio)
    df_train = df.iloc[:split_idx]
    df_test = df.iloc[split_idx:]

    print("Fitting feature & target scalers using training set only...")
    feature_scaler = MinMaxScaler()
    target_scaler = MinMaxScaler()

    feature_scaler.fit(df_train[feature_list])
    target_scaler.fit(df_train[['error_x_km', 'error_y_km', 'error_z_km']])

    scaled_train_f = feature_scaler.transform(df_train[feature_list])
    scaled_train_t = target_scaler.transform(df_train[['error_x_km','error_y_km','error_z_km']])

    scaled_test_f = feature_scaler.transform(df_test[feature_list])
    scaled_test_t = target_scaler.transform(df_test[['error_x_km','error_y_km','error_z_km']])

    def create_seq(X, Y, lookback):
        seq_X, seq_Y = [], []
        for i in range(len(X) - lookback):
            seq_X.append(X[i:i+lookback])
            seq_Y.append(Y[i+lookback])
        return np.array(seq_X), np.array(seq_Y)

    print(f"Creating sequences with lookback={lookback}...")
    X_test, y_test_scaled = create_seq(scaled_test_f, scaled_test_t, lookback)

    print(f"X_test shape = {X_test.shape}")
    print(f"y_test_scaled shape = {y_test_scaled.shape}")

    print(f"Loading trained model: {model_path}")
    model = load_model(model_path)

    print("Predicting...")
    y_pred_scaled = model.predict(X_test, verbose=0)

    y_pred = target_scaler.inverse_transform(y_pred_scaled)
    y_test = target_scaler.inverse_transform(y_test_scaled)

    baseline_error = np.linalg.norm(y_test, axis=1)
    hybrid_error = np.linalg.norm(y_test - y_pred, axis=1)

    mean_baseline = baseline_error.mean()
    mean_hybrid = hybrid_error.mean()
    improvement = ((mean_baseline - mean_hybrid) / mean_baseline) * 100

    print("\n---------- FINAL RESULTS ----------")
    print(f"Baseline SGP4-only Mean Error: {mean_baseline:.3f} km")
    print(f"Hybrid Model Mean Error:        {mean_hybrid:.3f} km")
    print(f"Improvement:                    {improvement:.2f}%")
    print("-----------------------------------\n")

    df_out = pd.DataFrame({
        'true_x': y_test[:,0],
        'true_y': y_test[:,1],
        'true_z': y_test[:,2],

        'pred_x': y_pred[:,0],
        'pred_y': y_pred[:,1],
        'pred_z': y_pred[:,2],

        'baseline_error_km': baseline_error,
        'hybrid_error_km': hybrid_error
    })

    df_out.to_csv(output_csv_path, index=False)
    print(f"Saved detailed errors to {output_csv_path}")

    return mean_baseline, mean_hybrid, improvement


In [20]:
def generate_feature_combinations(all_features):
    combos = []
    for k in range(1, len(all_features) + 1):
        for subset in itertools.combinations(all_features, k):
            combos.append(list(subset))
    return combos

In [21]:
from itertools import combinations
import os 

def run_all_combinations(
    satellites,
    df_parsed,
    all_features, 
    model_save_path_template,
    output_csv_path_template,
    datasets_output_path_template,
    k_values,
    lookback=30,
    train_split_ratio=0.8
):

    MODEL_DIR = os.path.dirname(model_save_path_template.format("temp"))
    RESULTS_DIR = os.path.dirname(output_csv_path_template.format("temp"))
    DATASETS_DIR = os.path.dirname(datasets_output_path_template.format("temp")) 
    if MODEL_DIR and not os.path.exists(MODEL_DIR):
        os.makedirs(MODEL_DIR)

    if RESULTS_DIR and not os.path.exists(RESULTS_DIR):
        os.makedirs(RESULTS_DIR)

    if DATASETS_DIR and not os.path.exists(DATASETS_DIR):
        os.makedirs(DATASETS_DIR)
    # ------------------------------------------------------------------------------------------

    generator = HybridDatasetGenerator(satellites, df_parsed)
    results = []

    for k in k_values:

        print("\n===================================================\n")
        print(f"Generating all combinations of size k = {k}")
        print("\n===================================================\n")

        feature_combos = list(combinations(all_features, k))

        for feats in feature_combos:

            print("\n===================================================\n")
            print(f"Processing features: {feats}")
            print("\n===================================================\n")

            hybrid_df = generator.generate_single(feats) 

            temp_hybrid_path = datasets_output_path_template.format('_'.join(feats)) 
            hybrid_df.to_csv(temp_hybrid_path, index=False)

            model_input_features = list(feats) + ['delta_t_days']
            model_input_features_sorted_str = "_".join(sorted(model_input_features))


            model_path_filename = model_save_path_template.format(model_input_features_sorted_str)
            model_name_prefix = model_input_features_sorted_str 

            train_loss, saved_model = train_hybrid_lstm(
                csv_path=temp_hybrid_path,
                feature_list=model_input_features, 
                target_list=['error_x_km','error_y_km','error_z_km'],
                lookback=lookback,
                model_name_prefix=model_name_prefix 
            )

            out_csv = output_csv_path_template.format(model_input_features_sorted_str)

            metrics = evaluate_model(
                feature_list=model_input_features, 
                hybrid_csv_path=temp_hybrid_path,
                model_path=saved_model,
                output_csv_path=out_csv,
                lookback=lookback,
                train_split_ratio=train_split_ratio
            )

            results.append({
                "features": tuple(model_input_features),
                "train_loss": train_loss,
                "mean_baseline_error": metrics[0],
                "mean_hybrid_error": metrics[1],
                "improvement_percent": metrics[2],
                "model_file": saved_model,
                "hybrid_data_file": temp_hybrid_path
            })

    return results


In [22]:

import os
from itertools import combinations
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler 
import numpy as np 
from skyfield.api import load 

MODEL_DIR = "models"
RESULTS_DIR = "results"
DATASETS_DIR = "datasets"

if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)
    print(f"Created directory: {MODEL_DIR}")

if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)
    print(f"Created directory: {RESULTS_DIR}")

if not os.path.exists(DATASETS_DIR):
    os.makedirs(DATASETS_DIR)
    print(f"Created directory: {DATASETS_DIR}")

FEATURES_FOR_COMBINATION = FEATURES_TO_USE 
ts = load.timescale()
satellites = load.tle_file(TLE_FILENAME)
print(f"Loaded {len(satellites)} satellites from {TLE_FILENAME}")

results = run_all_combinations(
    satellites=satellites,
    df_parsed=df_parsed,
    all_features=FEATURES_FOR_COMBINATION, 
    model_save_path_template=f"{MODEL_DIR}/model_{{}}.h5",
    output_csv_path_template=f"{RESULTS_DIR}/results_{{}}.csv",
    datasets_output_path_template=f"{DATASETS_DIR}/hybrid_temp_{{}}.csv",
    k_values=[4,5,6,7],
    lookback=30,
    train_split_ratio=0.8
)

df_results = pd.DataFrame(results)
df_results.to_csv("final_feature_combination_summary.csv", index=False)


Created directory: datasets
Loaded 4460 satellites from tle_history_38054.txt


Generating all combinations of size k = 4




Processing features: ('mean_motion_rev_day', 'eccentricity', 'bstar_drag', 'inclination_deg')



=== Training model for features: ['mean_motion_rev_day', 'eccentricity', 'bstar_drag', 'inclination_deg', 'delta_t_days'] ===
Loading dataset: datasets/hybrid_temp_mean_motion_rev_day_eccentricity_bstar_drag_inclination_deg.csv
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


111/111 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0588 - val_loss: 0.0126
Epoch 2/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 0.0063 - val_loss: 0.0139
Epoch 3/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.0054 - val_loss: 0.0122
Epoch 4/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - loss: 0.0049 - val_loss: 0.0127
Epoch 5/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0047 - val_loss: 0.0123
Epoch 6/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0042 - val_loss: 0.0109
Epoch 7/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0030 - val_loss: 0.0104
Epoch 8/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0026 - val_loss: 0.0102
Epoch 9/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0022 - val_loss: 0.0101
Epoch 10/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0021 - val_loss: 0.0102
Epoch 11/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0020 - val_loss: 0.0101
Epoch 12/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms

Saved model as bstar_drag_delta_t_days_eccentricity_inclination_deg_mean_motion_rev_day_mean_motion_rev_day_eccentricity_bstar_drag_inclination_deg_delta_t_days.h5
Final val loss = 0.009583414532244205

 Evaluating Model Using Features: ['mean_motion_rev_day', 'eccentricity', 'bstar_drag', 'inclination_deg', 'delta_t_days']
Loading hybrid dataset: datasets/hybrid_temp_mean_motion_rev_day_eccentricity_bstar_drag_inclination_deg.csv
Fitting feature & target scalers using training set only...
Creating sequences with lookback=30...
X_test shape = (862, 30, 5)
y_test_scaled shape = (862, 3)
Loading trained model: bstar_drag_delta_t_days_eccentricity_inclination_deg_mean_motion_rev_day_mean_motion_rev_day_eccentricity_bstar_drag_inclination_deg_delta_t_days.h5


Predicting...

---------- FINAL RESULTS ----------
Baseline SGP4-only Mean Error: 1.660 km
Hybrid Model Mean Error:        2.667 km
Improvement:                    -60.70%
-----------------------------------

Saved detailed errors to results/results_bstar_drag_delta_t_days_eccentricity_inclination_deg_mean_motion_rev_day.csv


Processing features: ('mean_motion_rev_day', 'eccentricity', 'bstar_drag', 'raan_deg')



=== Training model for features: ['mean_motion_rev_day', 'eccentricity', 'bstar_drag', 'raan_deg', 'delta_t_days'] ===
Loading dataset: datasets/hybrid_temp_mean_motion_rev_day_eccentricity_bstar_drag_raan_deg.csv
Epoch 1/100


...

Saved model as arg_perigee_deg_bstar_drag_delta_t_days_eccentricity_inclination_deg_mean_anomaly_deg_raan_deg_eccentricity_bstar_drag_inclination_deg_raan_deg_arg_perigee_deg_mean_anomaly_deg_delta_t_days.h5
Final val loss = 0.009557796642184258

 Evaluating Model Using Features: ['eccentricity', 'bstar_drag', 'inclination_deg', 'raan_deg', 'arg_perigee_deg', 'mean_anomaly_deg', 'delta_t_days']
Loading hybrid dataset: datasets/hybrid_temp_eccentricity_bstar_drag_inclination_deg_raan_deg_arg_perigee_deg_mean_anomaly_deg.csv
Fitting feature & target scalers using training set only...


Creating sequences with lookback=30...
X_test shape = (862, 30, 7)
y_test_scaled shape = (862, 3)
Loading trained model: arg_perigee_deg_bstar_drag_delta_t_days_eccentricity_inclination_deg_mean_anomaly_deg_raan_deg_eccentricity_bstar_drag_inclination_deg_raan_deg_arg_perigee_deg_mean_anomaly_deg_delta_t_days.h5
Predicting...

---------- FINAL RESULTS ----------
Baseline SGP4-only Mean Error: 1.660 km
Hybrid Model Mean Error:        3.189 km
Improvement:                    -92.18%
-----------------------------------

Saved detailed errors to results/results_arg_perigee_deg_bstar_drag_delta_t_days_eccentricity_inclination_deg_mean_anomaly_deg_raan_deg.csv


Generating all combinations of size k = 7




Processing features: ('mean_motion_rev_day', 'eccentricity', 'bstar_drag', 'inclination_deg', 'raan_deg', 'arg_perigee_deg', 'mean_anomaly_deg')



=== Training model for features: ['mean_motion_rev_day', 'eccentricity', 'bstar_drag', 'inclination_deg', 'raan_deg', 'arg_perigee_deg', 'mea

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.1033 - val_loss: 0.0132
Epoch 2/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0090 - val_loss: 0.0123
Epoch 3/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0058 - val_loss: 0.0125
Epoch 4/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0050 - val_loss: 0.0127
Epoch 5/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0049 - val_loss: 0.0123
Epoch 6/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0049 - val_loss: 0.0119
Epoch 7/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0040 - val_loss: 0.0108
Epoch 8/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0033 - val_loss: 0.0105
Epoch 9/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0029 - val_loss: 0.0103
Epoch 10/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0023 - val_loss: 0.0104
Epoch 11/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0023 - val_loss: 0.0108
Epoch 12/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/

Saved model as arg_perigee_deg_bstar_drag_delta_t_days_eccentricity_inclination_deg_mean_anomaly_deg_mean_motion_rev_day_raan_deg_mean_motion_rev_day_eccentricity_bstar_drag_inclination_deg_raan_deg_arg_perigee_deg_mean_anomaly_deg_delta_t_days.h5
Final val loss = 0.010071140713989735

 Evaluating Model Using Features: ['mean_motion_rev_day', 'eccentricity', 'bstar_drag', 'inclination_deg', 'raan_deg', 'arg_perigee_deg', 'mean_anomaly_deg', 'delta_t_days']
Loading hybrid dataset: datasets/hybrid_temp_mean_motion_rev_day_eccentricity_bstar_drag_inclination_deg_raan_deg_arg_perigee_deg_mean_anomaly_deg.csv
Fitting feature & target scalers using training set only...
Creating sequences with lookback=30...
X_test shape = (862, 30, 8)
y_test_scaled shape = (862, 3)
Loading trained model: arg_perigee_deg_bstar_drag_delta_t_days_eccentricity_inclination_deg_mean_anomaly_deg_mean_motion_rev_day_raan_deg_mean_motion_rev_day_eccentricity_bstar_drag_inclination_deg_raan_deg_arg_perigee_deg_mean_an